# Interferometer trajectory plotting

This notebook demonstrates `aispy.trajectory`: loading snapshot data from an
ais++ trajectory run and reconstructing smooth spacetime diagrams.

### How it works

Setting `printtrajectory 1` in the `.aisi` file causes ais++ to record
the position and velocity of every wavepacket at each pulse boundary
(before and after each pulse, plus the initial state and detection time).
Between snapshots the motion is analytic free-flight under gravity:
$$z(t) = z_0 + v_z(t_0)\,(t-t_0) - \tfrac{1}{2}g(t-t_0)^2$$
so the trajectory can be reconstructed at arbitrary resolution without
approximation for `linear_pot`.

### Intended use

`printtrajectory 1` is designed for **single-atom** (`natoms 1`) runs to
visualise the interferometer geometry.  Running with many atoms produces
very large files.

### Setup

```bash
pip install aispy   # or: pip install -e /path/to/aispy

# Generate the trajectory (from the aispp/trajectory-plots branch):
cd /path/to/aispp/examples
ais++ -i input-files/TRAJ_MZ_N1.aisi -o output-files/TRAJ_MZ_N1.h5
# This produces output-files/TRAJ_MZ_N1_TRAJ.h5
```
A pre-generated `TRAJ_MZ_N1_TRAJ.h5` is included in the aispp repo so you
can run this notebook without building ais++.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, '..')

from aispy.trajectory import load_trajectory, reconstruct_trajectories, \
                              plot_trajectory, plot_arm_separation

try:
    plt.style.use('seaborn-v0_8-ticks')
except OSError:
    plt.style.use('seaborn-ticks')
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9,
    'axes.labelsize': 9, 'savefig.dpi': 150,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'legend.frameon': False,
})

# Path to the pre-generated trajectory file (or your own run)
TRAJ_FILE = '../../aispp/examples/output-files/TRAJ_MZ_N1_TRAJ.h5'

## 1. Inspect the raw snapshot data

In [ ]:
traj = load_trajectory(TRAJ_FILE)

snap_times  = traj['snapshot_times']
snap_labels = traj['snapshot_labels']

print(f"Snapshots : {len(snap_times)}")
print(f"Records   : {len(traj['paths'])}")
print()
print(f"{'snap':>4}  {'label':>20}  {'t [s]':>8}")
print('-' * 38)
for i, (t, lbl) in enumerate(zip(snap_times, snap_labels)):
    print(f"{i:>4}  {lbl:>20}  {t:>8.4f}")

print()
# Show wavepackets at the mirror snapshot
idx = snap_labels.index('pulse_1_start') if 'pulse_1_start' in snap_labels else 2
mask = np.asarray(traj['snapshot_idx']) == idx
print(f"Wavepackets at snapshot {idx} ({snap_labels[idx]}, t={snap_times[idx]:.4f} s):")
for path, state, amp, z in zip(
    np.asarray(traj['paths'])[mask],
    traj['states'][mask],
    traj['amplitudes'][mask],
    traj['positions'][mask, 2]):
    print(f"  path={path:>6}  state={state}  amp={amp:.3f}  z={z:.4f} m")
    
# Arm separation at mirror:
z_vals = traj['positions'][mask, 2]
if len(z_vals) >= 2:
    print(f"\nArm separation at mirror: Δz = {abs(z_vals[1]-z_vals[0])*100:.2f} cm")

## 2. Spacetime diagram and transverse trajectory

The classic interferometer spacetime diagram: $z$ vs $t$ shows the
two arms separating after the first beam splitter, being redirected
by the mirror pulse, and recombining at the final beam splitter.
Grey bands mark the pulse durations (barely visible at this scale —
pulse duration $\tau \approx 0.25\,\text{ms} \ll T = 2.225\,\text{s}$).

In [ ]:
fig, (ax_z, ax_x) = plot_trajectory(traj, figsize=(8, 5.5), n_interp=300)
plt.savefig('trajectory_spacetime.png', bbox_inches='tight')
plt.show()

## 3. Arm separation vs time

The area enclosed by the two arms in the spacetime diagram is
proportional to $\hbar k_z g T^2$, the MZ gravitational phase.

In [ ]:
fig_sep, ax_sep = plot_arm_separation(traj, figsize=(7, 3))
plt.savefig('trajectory_arm_separation.png', bbox_inches='tight')
plt.show()

# Cross-check: max separation should occur at t = T
T = 2.225   # s
hbar = 1.054571817e-34
kz   = 8.996e6   # rad/m  (approximate)
m    = 86.909 * 1.66054e-27   # kg Sr-87
dz_theory = hbar * kz / m * T   # arm separation at t=T
print(f"Theoretical max arm separation: Δz = ℏkz/m × T = {dz_theory*100:.2f} cm")

## 4. Reconstruct manually and inspect

Access the raw reconstructed data for custom analysis or plotting.

In [ ]:
smooth = reconstruct_trajectories(traj, atom_idx=0, n_interp=500)

print("Reconstructed paths:")
for path, d in smooth.items():
    print(f"  {path:>6}  state={d['state']}  amp={d['amplitude']:.3f}  "
          f"t=[{d['t'][0]:.4f}, {d['t'][-1]:.4f}] s  "
          f"z=[{d['z'][0]*100:.1f}, {d['z'][-1]*100:.1f}] cm  "
          f"({len(d['t'])} points)")

# Example: compute phase accumulated along each arm from the kinematic action
# (separate from the laser phase — stored in ais++ as PhaseDouble)
# Here we just show the z-velocity of the two main arms at the mirror
print()
for path in ['00', '01']:
    if path in smooth:
        d = smooth[path]
        # midpoint index
        mid = len(d['t']) // 2
        vz_approx = np.gradient(d['z'], d['t'])
        print(f"  vz at midpoint for path '{path}': {vz_approx[mid]:.4f} m/s")

## 5. Overlay multiple atoms — effect of initial conditions

Run ais++ with `natoms > 1` (e.g., a small psgrid with a few representative
atoms) and overlay their trajectories.  This shows how different initial
positions and velocities change the spacetime path.

```bash
# Example: 3×1 grid  (x0 = -100, 0, +100 µm;  vx0 = 0 fixed)
python build_psgrid_input.py --nx 3 --nvx 1 --stem TRAJ_3ATOMS
# then add  printtrajectory 1  to the generated .aisi and run ais++
```

In [ ]:
# Placeholder — uncomment and adjust path once you have a multi-atom trajectory file
# MULTI_TRAJ_FILE = '../../aispp/examples/output-files/TRAJ_3ATOMS_TRAJ.h5'
# if os.path.exists(MULTI_TRAJ_FILE):
#     traj_multi = load_trajectory(MULTI_TRAJ_FILE)
#     n_atoms = int(np.asarray(traj_multi['atom_indices']).max()) + 1
#     fig, (az, ax) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
#     colors = plt.cm.viridis(np.linspace(0, 1, n_atoms))
#     for ai, col in zip(range(n_atoms), colors):
#         smooth_i = reconstruct_trajectories(traj_multi, atom_idx=ai)
#         for path, d in smooth_i.items():
#             az.plot(d['t'], d['z']*100, color=col, lw=0.8, alpha=0.7)
#             ax.plot(d['t'], d['x']*1e3, color=col, lw=0.8, alpha=0.7)
#     az.set_ylabel('z [cm]'); ax.set_ylabel('x [mm]'); ax.set_xlabel('t [s]')
#     plt.tight_layout(); plt.show()
# else:
print("Multi-atom trajectory file not found — run ais++ with a psgrid + printtrajectory 1.")